In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

In [2]:
from pathlib import Path

def _complaints_csv() -> Path:
    for d in _notebook_dir_candidates():
        p = d / "complaints.csv"
        if p.is_file() and p.stat().st_size > 0:
            return p
    raise FileNotFoundError("complaints.csv missing or empty next to the notebook or cwd.")


def _notebook_dir_candidates():
    out = []
    try:
        from IPython import get_ipython

        ip = get_ipython()
        nb = ip.user_ns.get("__vsc_ipynb_file__") if ip else None
        if nb:
            root = Path(nb).resolve().parent
            out.extend([root, *root.parents])
    except Exception:
        pass
    here = Path.cwd().resolve()
    out.extend([here, *here.parents])
    seen = set()
    for d in out:
        if d not in seen:
            seen.add(d)
            yield d


df = pd.read_csv(_complaints_csv())

In [3]:
df.head()

,text,category
0,The road in front of our colony has been full ...,road
1,There has been no water supply in our area sin...,water
2,Power cut is going on continuously for the pas...,electricity
3,Garbage has not been collected from our street...,garbage
4,The main road near the bus stop is completely ...,road


In [4]:
#renaming the column
df.rename(columns={'text':'complaint'},inplace=True)
df.head()

,complaint,category
0,The road in front of our colony has been full ...,road
1,There has been no water supply in our area sin...,water
2,Power cut is going on continuously for the pas...,electricity
3,Garbage has not been collected from our street...,garbage
4,The main road near the bus stop is completely ...,road


Cleaning the datset(basic nlp)

In [5]:
#1.tolowercase
df["cleaned_complaint"] = df["complaint"].str.lower().fillna("")
df.head()


,complaint,category,cleaned_complaint
0,The road in front of our colony has been full ...,road,the road in front of our colony has been full ...
1,There has been no water supply in our area sin...,water,there has been no water supply in our area sin...
2,Power cut is going on continuously for the pas...,electricity,power cut is going on continuously for the pas...
3,Garbage has not been collected from our street...,garbage,garbage has not been collected from our street...
4,The main road near the bus stop is completely ...,road,the main road near the bus stop is completely ...


In [6]:
#2.remove punctuation like !,.,?, etc
df["cleaned_complaint"] = df["cleaned_complaint"].str.replace(r"[^\w\s]", "", regex=True).fillna("")
df.head()


,complaint,category,cleaned_complaint
0,The road in front of our colony has been full ...,road,the road in front of our colony has been full ...
1,There has been no water supply in our area sin...,water,there has been no water supply in our area sin...
2,Power cut is going on continuously for the pas...,electricity,power cut is going on continuously for the pas...
3,Garbage has not been collected from our street...,garbage,garbage has not been collected from our street...
4,The main road near the bus stop is completely ...,road,the main road near the bus stop is completely ...


In [7]:
# 3. Stopwords — golden rule for civic complaints
# Keep: negation (no, not) | time (since, days, week, …) | severity (urgent, severe)
# Remove: fillers (the, is, has, been, …) plus NLTK English high-frequency words
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("stopwords", quiet=True)
nltk.download("punkt_tab", quiet=True)

_en_stop = set(stopwords.words("english"))

# Never remove these (even if they appear in NLTK or filler sets)
_KEEP = frozenset({
    "no",
    "not",
    "nor",
    "but",
    "since",
    "day",
    "days",
    "week",
    "weeks",
    "urgent",
    "severe",
    "severely",
    "over",
})

# Explicit fillers to always drop (subset already in NLTK; listed for clarity)
_FILLERS = frozenset({
    "the",
    "a",
    "an",
    "is",
    "am",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "has",
    "have",
    "had",
    "having",
})

STOPWORDS = (_en_stop | _FILLERS) - _KEEP


def _remove_stopwords(text: str) -> str:
    if not isinstance(text, str):
        text = "" if text is None or (isinstance(text, float) and np.isnan(text)) else str(text)
    text = text.strip().lower()
    if not text or text == "nan":
        return ""
    tokens = word_tokenize(text)
    kept = [w for w in tokens if w not in STOPWORDS]
    return " ".join(kept)


df["cleaned_complaint"] = df["cleaned_complaint"].fillna("").map(_remove_stopwords)
df.head()


,complaint,category,cleaned_complaint
0,The road in front of our colony has been full ...,road,road front colony full potholes over week
1,There has been no water supply in our area sin...,water,no water supply area since yesterday morning
2,Power cut is going on continuously for the pas...,electricity,power cut going continuously past 6 hours loca...
3,Garbage has not been collected from our street...,garbage,garbage not collected street last 4 days
4,The main road near the bus stop is completely ...,road,main road near bus stop completely broken caus...


In [8]:
# 4. Lemmatization — WordNet + POS tags (verbs/adjectives lemmatize better than noun-only)
import nltk
from nltk import pos_tag
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

_lemmatizer = WordNetLemmatizer()


def _penn_to_wordnet(tag: str):
    if tag.startswith("J"):
        return wn.ADJ
    if tag.startswith("V"):
        return wn.VERB
    if tag.startswith("N"):
        return wn.NOUN
    if tag.startswith("R"):
        return wn.ADV
    return wn.NOUN


def _lemmatize_text(text: str) -> str:
    if not isinstance(text, str):
        text = "" if text is None or (isinstance(text, float) and np.isnan(text)) else str(text)
    text = text.strip()
    if not text or text.lower() == "nan":
        return ""
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens, lang="eng")
    lemmas = [_lemmatizer.lemmatize(w.lower(), _penn_to_wordnet(t)) for w, t in tagged]
    return " ".join(lemmas)


df["cleaned_complaint"] = df["cleaned_complaint"].fillna("").map(_lemmatize_text)
df.head()


,complaint,category,cleaned_complaint
0,The road in front of our colony has been full ...,road,road front colony full pothole over week
1,There has been no water supply in our area sin...,water,no water supply area since yesterday morning
2,Power cut is going on continuously for the pas...,electricity,power cut go continuously past 6 hour locality
3,Garbage has not been collected from our street...,garbage,garbage not collect street last 4 day
4,The main road near the bus stop is completely ...,road,main road near bus stop completely broken caus...


In [9]:
#lets compare  the sentences before and after cleaning
df[["complaint", "cleaned_complaint"]].head(10)


,complaint,cleaned_complaint
0,The road in front of our colony has been full ...,road front colony full pothole over week
1,There has been no water supply in our area sin...,no water supply area since yesterday morning
2,Power cut is going on continuously for the pas...,power cut go continuously past 6 hour locality
3,Garbage has not been collected from our street...,garbage not collect street last 4 day
4,The main road near the bus stop is completely ...,main road near bus stop completely broken caus...
5,Tap water supply has been completely dry in ou...,tap water supply completely dry building since...
6,Lights on our street have not been working for...,light street not work 3 night straight
7,Trash bins near the park are overflowing and n...,trash bin near park overflow no one come clear
8,A massive pothole near the school entrance has...,massive pothole near school entrance injure tw...
9,Water pipeline near my house has been leaking ...,water pipeline near house leak 2 day nobody fix


In [10]:
#lets see one complaint before and after cleaning
print(f"Before cleaning: {df['complaint'].iloc[0]}")
print(f"After cleaning: {df['cleaned_complaint'].iloc[0]}")

#lets see the length of the complaint before and after cleaning
print(f"Length of complaint before cleaning: {len(df['complaint'].iloc[0])}")
print(f"Length of complaint after cleaning: {len(df['cleaned_complaint'].iloc[0])}")


Before cleaning: The road in front of our colony has been full of potholes for over a week now
After cleaning: road front colony full pothole over week
Length of complaint before cleaning: 77
Length of complaint after cleaning: 40


In [16]:
# TF-IDF on cleaned_complaint (no sklearn stop_words — you already removed stopwords with your rules)
from sklearn.feature_extraction.text import TfidfVectorizer

_text = df["cleaned_complaint"].fillna("").astype(str)

tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words=None,
    ngram_range=(1, 2),
    min_df=1,
    # Letters only (length >= 2) so features are words, not tokens like "10", "2hour"
    token_pattern=r"(?u)\b[a-z]{2,}\b",
)
X = tfidf_vectorizer.fit_transform(_text)

# Feature names = one per column, in alphabetical order (sklearn vocabulary sort)
feature_names = tfidf_vectorizer.get_feature_names_out()
print("First 5 feature names:", list(feature_names[:5]))

# Example complaint keywords (only listed if they appear in this corpus)
_sample = ["area", "garbage", "pothole", "road", "water"]
print("Sample domain words in vocab:", [w for w in _sample if w in set(feature_names)])

# First 5 rows as dense only for printing (X stays sparse for models / large data)
print(X[:5].toarray())
print(X.shape)


First 5 feature names: ['abandon', 'abandon reflect', 'able', 'able pas', 'abnormally']
Sample domain words in vocab: ['area', 'garbage', 'pothole', 'road', 'water']
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.38838608 0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
(250, 1000)


In [18]:
# Train / test split — stratify so road / water / electricity / garbage stay balanced in each set
from sklearn.model_selection import train_test_split

y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("y_train:\n", y_train.value_counts())
print("y_test:\n", y_test.value_counts())


X_train: (200, 1000)  X_test: (50, 1000)
y_train:
 category
garbage        50
water          50
electricity    50
road           50
Name: count, dtype: int64
y_test:
 category
electricity    13
water          13
road           12
garbage        12
Name: count, dtype: int64


In [24]:
# Multinomial Naive Bayes — fits well with TF-IDF / word counts (non-negative features)
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB(alpha=1.0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Class probabilities (rows = samples, columns = model.classes_ order)
y_proba = model.predict_proba(X_test)

# Confidence = predicted class probability (max prob per row)
confidence_score = y_proba.max(axis=1)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

import pandas as pd

proba_df = pd.DataFrame(y_proba, columns=model.classes_)
proba_df["predicted"] = y_pred
proba_df["confidence"] = confidence_score
proba_df["true"] = y_test.values
proba_df.head()


Test accuracy: 0.94
              precision    recall  f1-score   support

 electricity       0.92      0.92      0.92        13
     garbage       1.00      1.00      1.00        12
        road       0.92      1.00      0.96        12
       water       0.92      0.85      0.88        13

    accuracy                           0.94        50
   macro avg       0.94      0.94      0.94        50
weighted avg       0.94      0.94      0.94        50



,electricity,garbage,road,water,predicted,confidence,true
0,0.278749,0.172485,0.340538,0.208228,road,0.340538,road
1,0.403982,0.143337,0.202161,0.250519,electricity,0.403982,electricity
2,0.150562,0.390055,0.262490,0.196893,garbage,0.390055,garbage
3,0.291531,0.236923,0.194661,0.276885,electricity,0.291531,electricity
4,0.351832,0.148274,0.156621,0.343273,electricity,0.351832,electricity


In [25]:
#let us test with 3 inputs
# Let's test the trained model with 3 custom complaint inputs
test_complaints = [
    "Water supply has been interrupted for two days and no help has arrived.",
    "Electrical wires are hanging loose near our market and pose a danger.",
    "Garbage has not been collected from the street and the bins are overflowing."
]

# Transform the test complaints using the same vectorizer as training
test_features = tfidf_vectorizer.transform(test_complaints)

# Predict categories and confidence scores
test_pred = model.predict(test_features)
test_proba = model.predict_proba(test_features)
test_confidence = test_proba.max(axis=1)

for i, complaint in enumerate(test_complaints):
    print(f"\nComplaint: {complaint}")
    print(f"Predicted Category: {test_pred[i]} (Confidence: {test_confidence[i]:.4f})")


Complaint: Water supply has been interrupted for two days and no help has arrived.
Predicted Category: water (Confidence: 0.8081)

Complaint: Electrical wires are hanging loose near our market and pose a danger.
Predicted Category: road (Confidence: 0.3350)

Complaint: Garbage has not been collected from the street and the bins are overflowing.
Predicted Category: garbage (Confidence: 0.5692)
